In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Ready!')
else:
    print('NO GPU — go to Runtime -> Change runtime type -> T4 GPU -> Save')

GPU: Tesla T4
Ready!


In [ ]:
import subprocess, os

print('Installing dependencies...')
subprocess.run('pip install -q opencv-python-headless scipy', shell=True)
print('Done installing')

print('\nDownloading RAFT...')
if not os.path.exists('RAFT'):
    os.system('git clone -q https://github.com/princeton-vl/RAFT.git')
    os.system('cd RAFT && bash download_models.sh')
    print('RAFT downloaded')
else:
    print('RAFT already present')

print('\nRAFT models available:')
print(os.listdir('RAFT/models'))

Installing dependencies...
Done installing

RAFT downloaded

RAFT models available:
['raft-kitti.pth', 'raft-sintel.pth', 'raft-things.pth', 'raft-chairs.pth', 'raft-small.pth']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

Mounted at /content/drive
Google Drive mounted!


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

['20211125_111213.jpg', '20211118_130922.jpg', '20221207_103154 (1).jpg', 'Screenshot_20230403-131840_Chrome (2).jpg', 'Screenshot_20230403-131840_Chrome (1).jpg', 'Screenshot_20230403-131840_Chrome.jpg', 'IMG-20230403-WA0010.jpg', 'IMG-20230403-WA0013 (3).jpg', 'IMG-20230403-WA0013 (2).jpg', 'IMG-20230403-WA0013 (1).jpg', 'IMG-20230403-WA0013.jpg', 'IMG_0656.png', 'IMG_7587_Original.jpeg', 'IMG_8753.jpeg', 'IMG_8153_Original.jpeg', 'Student Weight Survey Analysis', 'Udaipur trip 2025', 'Colab Notebooks', 'Andrew S. Tanenbaum - Computer Networks.pdf', 'sd', 'videos', 'Top MNIT Recruiter Project Ideas.gdoc']


In [ ]:
import os
videos = os.listdir('/content/drive/MyDrive/videos')
print(f'Found {len(videos)} files:')
for v in sorted(videos):
    print(' ', v)

Found 25 files:
  MVI_0788_VIS_OB.avi
  MVI_0788_VIS_OB_Borderless.avi
  MVI_0788_VIS_OB_FINAL.avi
  MVI_0788_VIS_OB_stabilized.avi
  MVI_0788_VIS_OB_stable.avi
  MVI_0789_VIS_OB.avi
  MVI_0789_VIS_OB_stabilized.avi
  MVI_0790_VIS_OB.avi
  MVI_0790_VIS_OB_stabilized.avi
  MVI_0792_VIS_OB.avi
  MVI_0792_VIS_OB_stabilized.avi
  MVI_0794_VIS_OB.avi
  MVI_0794_VIS_OB_stabilized.avi
  MVI_0795_VIS_OB.avi
  MVI_0795_VIS_OB_stabilized.avi
  MVI_0796_VIS_OB.avi
  MVI_0796_VIS_OB_stabilized.avi
  MVI_0797_VIS_OB.avi
  MVI_0797_VIS_OB_stabilized.avi
  MVI_0799_VIS_OB.avi
  MVI_0799_VIS_OB_stabilized.avi
  MVI_0801_VIS_OB.avi
  MVI_0801_VIS_OB_stabilized.avi
  MVI_0804_VIS_OB.avi
  MVI_0804_VIS_OB_stabilized.avi


In [ ]:
import sys, cv2, numpy as np, torch
from scipy.ndimage import uniform_filter1d

sys.path.insert(0, 'RAFT/core')
from raft import RAFT
from utils.utils import InputPadder

#TUNIGN PARAMETERS
SMOOTHING_RADIUS = 30
CROP_RATIO       = 0.07
FLOW_W, FLOW_H    = 960, 540
RAFT_MODEL_PATH  = 'RAFT/models/raft-sintel.pth'

device = torch.device('cuda')
print('Using device:', torch.cuda.get_device_name(0))

#LOADING RAFT ONCE
class Args:
    small = False; mixed_precision = False
    alternate_corr = False; dropout = 0.0
    def __contains__(self, item): return hasattr(self, item)
    def __getitem__(self, item): return getattr(self, item)

raft_model = RAFT(Args())
weights = torch.load(RAFT_MODEL_PATH, map_location=device)
raft_model.load_state_dict({k.replace('module.', ''): v for k, v in weights.items()})
raft_model = raft_model.to(device).eval()
print('RAFT model loaded')


def to_tensor(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float().unsqueeze(0).to(device)

def estimate_flow(f1, f2):
    s1 = cv2.resize(f1, (FLOW_W, FLOW_H), interpolation=cv2.INTER_AREA)
    s2 = cv2.resize(f2, (FLOW_W, FLOW_H), interpolation=cv2.INTER_AREA)
    with torch.no_grad():
        t1, t2 = to_tensor(s1), to_tensor(s2)
        padder = InputPadder(t1.shape)
        t1, t2 = padder.pad(t1, t2)
        _, flow = raft_model(t1, t2, iters=20, test_mode=True)
        flow = padder.unpad(flow)[0].permute(1, 2, 0).cpu().numpy()
    h, w = f1.shape[:2]
    flow = cv2.resize(flow, (w, h), interpolation=cv2.INTER_LINEAR)
    flow[..., 0] *= w / FLOW_W
    flow[..., 1] *= h / FLOW_H
    return flow

def flow_to_transform(flow):
    h, w = flow.shape[:2]
    step = 16
    ys, xs = np.mgrid[step//2:h:step, step//2:w:step]
    src = np.stack([xs.ravel(), ys.ravel()], axis=1).astype(np.float32)
    dx = flow[ys.ravel(), xs.ravel(), 0]
    dy = flow[ys.ravel(), xs.ravel(), 1]
    dst = src + np.stack([dx, dy], axis=1).astype(np.float32)
    M, _ = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    return M if M is not None else np.eye(2, 3, dtype=np.float32)

def get_cumulative(transforms):
    cum = [np.eye(2, 3, dtype=np.float64)]
    for M in transforms:
        prev3 = np.vstack([cum[-1], [0, 0, 1]])
        curr3 = np.vstack([M.astype(np.float64), [0, 0, 1]])
        cum.append((curr3 @ prev3)[:2, :])
    return cum

def smooth_trajectory(cum):
    arr = np.array(cum).reshape(-1, 6)
    size = 2 * SMOOTHING_RADIUS + 1
    out = np.zeros_like(arr)
    for i in range(6):
        out[:, i] = uniform_filter1d(arr[:, i], size=size)
    return out.reshape(-1, 2, 3)

def get_corrections(cum, scum):
    corr = []
    for o, s in zip(cum, scum):
        o3 = np.vstack([o, [0, 0, 1]])
        s3 = np.vstack([s, [0, 0, 1]])
        corr.append((s3 @ np.linalg.inv(o3))[:2, :].astype(np.float32))
    return corr

def apply_transform(frame, M, output_size):
    return cv2.warpAffine(
        frame, M, output_size,
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REPLICATE
    )

def crop_frame(frame, ratio):
    h, w = frame.shape[:2]
    dy, dx = int(h * ratio), int(w * ratio)
    return frame[dy:h-dy, dx:w-dx]

def stabilize_video(input_path, output_path):
    print(f'\n[video] Opening: {input_path}')
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'[video] {width}x{height} @ {fps}fps — {total} frames')

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    print(f'[video] Read {len(frames)} frames')

    print('[flow] Estimating optical flow...')
    transforms = []
    for i in range(len(frames) - 1):
        flow = estimate_flow(frames[i], frames[i+1])
        transforms.append(flow_to_transform(flow))
        if (i + 1) % 50 == 0 or i == 0:
            print(f'[flow] {i+1}/{len(frames)-1} pairs done')
    print('[flow] Done')

    print('[trajectory] Smoothing...')
    cum = get_cumulative(transforms)
    scum = smooth_trajectory(cum)
    corrections = get_corrections(cum, scum)

    print('[warp] Writing stabilized output...')
    test = crop_frame(frames[0], CROP_RATIO)
    out_h, out_w = test.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(output_path, fourcc, fps, (out_w, out_h))

    for i in range(len(frames)):
        warped = apply_transform(frames[i], corrections[i], (width, height))
        cropped = crop_frame(warped, CROP_RATIO)
        out.write(cropped)
        if (i + 1) % 100 == 0:
            print(f'[warp] {i+1}/{len(frames)} frames written')

    out.release()
    print(f'[done] Saved: {output_path} ({out_w}x{out_h})')

print('\nStabilizer function ready. Run the next cell to process videos.')

ModuleNotFoundError: No module named 'raft'

In [ ]:
import os

VIDEO_FOLDER = '/content/drive/MyDrive/videos'

# Change this filename to process a different video
INPUT_FILENAME = 'MVI_0788_VIS_OB.avi'

input_path = os.path.join(VIDEO_FOLDER, INPUT_FILENAME)
name_only = os.path.splitext(INPUT_FILENAME)[0]
output_path = os.path.join(VIDEO_FOLDER, f'{name_only}_stabilized.avi')

stabilize_video(input_path, output_path)


[video] Opening: /content/drive/MyDrive/videos/MVI_0788_VIS_OB.avi
[video] 1920x1080 @ 30.0fps — 299 frames
[video] Read 299 frames
[flow] Estimating optical flow...


/content/RAFT/core/raft.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/content/RAFT/core/raft.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/RAFT/core/raft.py:127: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args.mixed_precision):


[flow] 1/298 pairs done
[flow] 50/298 pairs done
[flow] 100/298 pairs done
[flow] 150/298 pairs done
[flow] 200/298 pairs done
[flow] 250/298 pairs done
[flow] Done
[trajectory] Smoothing...
[warp] Writing stabilized output...
[warp] 100/299 frames written
[warp] 200/299 frames written
[done] Saved: /content/drive/MyDrive/videos/MVI_0788_VIS_OB_stabilized.avi (1652x930)


In [ ]:
import os

VIDEO_FOLDER = '/content/drive/MyDrive/videos'

REMAINING_VIDEOS = [
    'MVI_0789_VIS_OB.avi',
    'MVI_0790_VIS_OB.avi',
    'MVI_0792_VIS_OB.avi',
    'MVI_0794_VIS_OB.avi',
    'MVI_0795_VIS_OB.avi',
    'MVI_0796_VIS_OB.avi',
    'MVI_0797_VIS_OB.avi',
    'MVI_0799_VIS_OB.avi',
    'MVI_0801_VIS_OB.avi',
    'MVI_0804_VIS_OB.avi',
]

for i, filename in enumerate(REMAINING_VIDEOS):
    print(f'\n{"="*70}')
    print(f'PROCESSING VIDEO {i+1}/{len(REMAINING_VIDEOS)}: {filename}')
    print(f'{"="*70}')

    input_path = os.path.join(VIDEO_FOLDER, filename)
    name_only = os.path.splitext(filename)[0]
    output_path = os.path.join(VIDEO_FOLDER, f'{name_only}_stabilized.avi')

    if not os.path.exists(input_path):
        print(f'[skip] {filename} not found, skipping')
        continue

    try:
        stabilize_video(input_path, output_path)
    except Exception as e:
        print(f'[error] Failed on {filename}: {e}')
        print('[continuing to next video]')
        continue

print('\n\nALL REMAINING VIDEOS PROCESSED!')
print('Check your Google Drive videos folder for all _stabilized.avi files')


PROCESSING VIDEO 1/10: MVI_0789_VIS_OB.avi

[video] Opening: /content/drive/MyDrive/videos/MVI_0789_VIS_OB.avi
[video] 1920x1080 @ 30.0fps — 279 frames
[video] Read 279 frames
[flow] Estimating optical flow...
[flow] 1/278 pairs done
[flow] 50/278 pairs done
[flow] 100/278 pairs done
[flow] 150/278 pairs done
[flow] 200/278 pairs done
[flow] 250/278 pairs done
[flow] Done
[trajectory] Smoothing...
[warp] Writing stabilized output...
[warp] 100/279 frames written
[warp] 200/279 frames written
[done] Saved: /content/drive/MyDrive/videos/MVI_0789_VIS_OB_stabilized.avi (1652x930)

PROCESSING VIDEO 2/10: MVI_0790_VIS_OB.avi

[video] Opening: /content/drive/MyDrive/videos/MVI_0790_VIS_OB.avi
[video] 1920x1080 @ 30.0fps — 600 frames
[video] Read 600 frames
[flow] Estimating optical flow...
[flow] 1/599 pairs done
[flow] 50/599 pairs done
[flow] 100/599 pairs done
[flow] 150/599 pairs done
[flow] 200/599 pairs done
[flow] 250/599 pairs done
[flow] 300/599 pairs done
[flow] 350/599 pairs done
